### Setup Data loader

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from notebooks.create_deepGP import ServiceGP, prepare_chained_data
import gpytorch
from agent.components.commons import ServiceFeatureMapping, ServiceType
from typing import List
import torch

from agent.components import RASK

torch.set_default_dtype(torch.float64)


In [2]:

class DynamicServiceChain(torch.nn.Module):
    def __init__(self, service_configs: List[ServiceFeatureMapping], num_inducing: int = 64):
        super().__init__()
        self.configs = service_configs
        self.gp_layers = torch.nn.ModuleList()
        self.likelihoods = torch.nn.ModuleList()

        for i, config in enumerate(service_configs):
            # The first service takes only raw features.
            # Subsequent services take raw features + 1 (previous service output).
            input_dims = len(config.feature_indices)
            if i > 0:
                input_dims += 1

            gp = ServiceGP(input_dims=input_dims, num_inducing=num_inducing)
            likelihood = gpytorch.likelihoods.GaussianLikelihood()

            self.gp_layers.append(gp)
            self.likelihoods.append(likelihood)

    def forward(self, x):
        dists = []
        last_output = None

        for i, gp in enumerate(self.gp_layers):
            # Extract raw features for this service
            indices = self.configs[i].feature_indices
            current_input = x[:, indices]

            # If not the first service, concat the sample from the previous GP
            if last_output is not None:
                current_input = torch.cat([current_input, last_output], dim=-1)

            dist = gp(current_input)
            dists.append(dist)

            # Reparameterization trick sample for the next layer's input
            last_output = dist.rsample().unsqueeze(-1)

        return tuple(dists)

In [3]:

raw_df = pd.read_csv("../statics/metrics_TSC_EXPLORE.csv")
converted_df = RASK.preprocess_data(raw_df)

QR_MAP = ServiceFeatureMapping(ServiceType.QR, [0, 1])
CV_MAP = ServiceFeatureMapping(ServiceType.CV, [2, 3, 4])
PC_MAP = ServiceFeatureMapping(ServiceType.PC, [5, 6])

repetitions = 30
# chain_definition = ([QR_MAP])
# chain_definition = ([QR_MAP] * repetitions) + ([CV_MAP] * repetitions) + ([PC_MAP] * repetitions)
chain_definition = ([QR_MAP] * repetitions)


train_loader, test_x, test_y, scaler_X = prepare_chained_data(converted_df, chain_definition, test_size=0.2)


### Setup the Structure


In [4]:

model = DynamicServiceChain(chain_definition, num_inducing=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Dynamic MLL setup
mlls = [
    gpytorch.mlls.VariationalELBO(model.likelihoods[i], model.gp_layers[i], num_data=len(train_loader.dataset))
    for i in range(len(chain_definition))
]

# Training Loop
model.train()
for epoch in range(500):
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        # Returns a tuple of distributions: (qr_dist, cv_dist, pc_dist)
        distributions = model(x_batch)

        # Calculate sum of losses dynamically
        loss = sum([-mlls[i](distributions[i], y_batch[:, i]) for i in range(len(mlls))])

        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")
    if epoch % 50 == 0:
        for i, likelihood in enumerate(model.likelihoods):
            print(f"Noise: {likelihood.noise.item():.4f}")

/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 0 | Loss: 41.3602
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Epoch 10 | Loss: 35.2116
Epoch 20 | Loss: 31.0134
Epoch 30 | Loss: 28.8472
Epoch 40 | Loss: 27.5064
Epoch 50 | Loss: 26.3852
Noise: 0.6424
Noise: 0.6119
Noise: 0.6221
Noise: 0.6174
Noise: 0.6076
Noise: 0.6229
Noise: 0.6177
Noise: 0.6050
Noise: 0.6025
Noise: 0.6062
Noise: 0.6040
Noise: 0.5995
Noise: 0.6024
Noise: 0.6043
Noise: 0.5956
Noise: 0.5989
Noise: 0.6071
Noise: 0.6007
Noise: 0.5973
Noise: 0.5933
Noise: 0.5898
Noise: 0.5951
Noise: 0.6026
Noise: 0.5916
Noise: 0.5968
Noise: 0.6027
Noise: 0.5993
Noise: 0.6006
Noise: 0.5906
Noise: 0.5953
Epoch 60 | 

KeyboardInterrupt: 

In [ ]:
model.eval()
all_samples = []
num_mc = 100  # Number of Monte Carlo simulations per data point
num_services = len(model.configs)

with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-3):
    for _ in range(num_mc):
        # 1. model(test_x) now returns a tuple of N distributions
        distributions = model(test_x)

        # 2. Sample from each distribution in the chain
        # We use a list comprehension to handle all N services dynamically
        samples = [dist.sample() for dist in distributions]

        # 3. Stack along the last dimension to keep (N_test, N_services)
        s = torch.stack(samples, dim=-1)
        all_samples.append(s)

    # 4. Stack all MC trials into a single NumPy array
    # Shape: (num_mc, num_test_samples, num_services)
    combined_samples = torch.stack(all_samples, dim=0).cpu().numpy()

print(f"Inference complete. Samples shape: {combined_samples.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculate the mean and std for EACH test case across simulations
# combined_samples shape: (num_mc, num_test_samples, num_services)
means_per_config = combined_samples.mean(axis=0)  # Shape: (N_test, N_services)
stds_per_config = combined_samples.std(axis=0)    # Shape: (N_test, N_services)

# 2. Average across all test configurations to get the global chain profile
global_means = means_per_config.mean(axis=0)      # Shape: (N_services,)
global_stds = stds_per_config.mean(axis=0)        # Shape: (N_services,)

# Extract service names for the X-axis labels
stages = range(len(chain_definition))

plt.figure(figsize=(12, 6))

# 3. Plot the Global Staircase
# fmt='-s' connects the dots to show the throughput "decay" or bottlenecking
plt.errorbar(stages, global_means, yerr=global_stds, fmt='-s', capsize=10,
             color='navy', ecolor='red', elinewidth=3, markersize=8,
             label='Average Chain Performance')

# 4. Annotate the Average Sigma for each stage
# for i, stage_name in enumerate(stages):
#     plt.text(i + 0.1, global_means[i], f'σ={global_stds[i]:.3f}',
#              va='bottom', fontweight='bold', color='red', fontsize=9)

plt.title(f"Global Throughput Bottleneck ({len(stages)} Stages)")
plt.ylabel("Average Scaled Throughput (0-1)")
plt.xlabel("Service Pipeline Sequence")

plt.grid(axis='y', alpha=0.3)
plt.legend(loc='upper right')

# 5. Shaded area showing Hardware Diversity (Best vs Worst config)
plt.fill_between(stages,
                 means_per_config.min(axis=0),
                 means_per_config.max(axis=0),
                 color='gray', alpha=0.1, label='Hardware Spread')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# --- MODEL VALIDATION ---
# 1. Get the mean prediction for each service across Monte Carlo samples
# combined_samples shape: (num_mc, N_test, num_services)
predicted_means = combined_samples.mean(axis=0)

# 2. Extract the ground truth (test_y)
actual_y = test_y.cpu().numpy()

# 3. Calculate Error for each service dynamically
# We iterate based on the number of services defined in the model
for i, config in enumerate(model.configs):
    # Calculate RMSE for the i-th service in the chain
    rmse = np.sqrt(np.mean((predicted_means[:, i] - actual_y[:, i]) ** 2))
    print(f"{config.service_type.value} Service RMSE: {rmse:.4f}")